In [0]:
# File location and type

customer_location = "/FileStore/tables/customer"
sales_location = "/FileStore/tables/sales"

file_type = "csv"


# The applied options are for CSV files. For other file types, these will be ignored.

customer_df = spark.read.format(file_type)\
.option("inferSchema", "true")\
.option("header", "true")\
.option("sep",",")\
.load(customer_location)
display(customer_df)

# The applied options are for CSV files. For other file types, these will be ignored.

sales_df = spark.read.format(file_type)\
.option("inferSchema", "true")\
.option("header", "true")\
.option("sep",",")\
.load(sales_location)
display(sales_df)

CustomerID,Name,Email,Phone,Address
4600,Rebecca Yu,dunndaryl @adkins .net,(892)6794750x59660,634 Tyler Hollow
Tammyville,"VT 90169""",null,null,null
5684,Gina Koch,brian61 @gaines .com,710.459.8498,044 Barnes Loaf Apt. 446
New Zacharytown,"WY 31707""",null,null,null
6647,James Diaz,larry89 @dean .net,1908774524x3160,7151 Cory Stravenue Apt. 817
South Joshua,"NC 39145""",null,null,null
4661,Mary Phillips,vbrown @parker-harrison .net,(732)6615896x309,55677 James Harbors Suite 047
North Rebecca,"AL 08689""",null,null,null
6093,Diamond Thompson,dhernandez @ramos .com,0012931539801x39200,073 Kerr Fords
Lake Bradleychester,"CT 53718""",null,null,null


CustomerID,SaleDate,SaleAmount
9341,2023-03-29,413.02
9341,2023-12-08,245.15
9829,2023-08-11,141.26
6647,2023-10-22,203.01
6647,2023-12-18,213.34
2453,2023-12-15,223.04
8689,2023-04-14,33.98
8689,2023-01-09,312.35
2453,2023-12-28,207.8
6647,2023-11-18,143.98


In [0]:
#1.Remove duplicate data in customer csv
customer_df1 = customer_df.dropDuplicates()
display(customer_df1)

CustomerID,Name,Email,Phone,Address
4600,Rebecca Yu,dunndaryl @adkins .net,(892)6794750x59660,634 Tyler Hollow
Catherinestad,"WV 51848""",null,null,null
6039,Dylan Grant,xsanchez @hotmail .com,1098985454,USNS Edwards
4661,Mary Phillips,vbrown @parker-harrison .net,(732)6615896x309,55677 James Harbors Suite 047
"FPO AP 23850""",null,null,null,null
Lake Bradleychester,"CT 53718""",null,null,null
Cooperton,"AZ 59040""",null,null,null
5684,Gina Koch,brian61 @gaines .com,710.459.8498,044 Barnes Loaf Apt. 446
9341,Michelle Carlson,sandersscott @white .com,0836792842x561,06748 Victor Harbors
Tammyville,"VT 90169""",null,null,null


In [0]:
#2.Clean email address and phone number of customer
from pyspark.sql.functions import regexp_replace
 
customer_df2 = customer_df1.withColumn("email", regexp_replace("email", "[^a-zA-Z0-9@._-]", ""))
display(customer_df2)
customer_df3 = customer_df2.withColumn("phone", regexp_replace("phone", "[^0-9]", ""))
display(customer_df3)

CustomerID,Name,email,Phone,Address
4600,Rebecca Yu,dunndaryl@adkins.net,(892)6794750x59660,634 Tyler Hollow
Catherinestad,"WV 51848""",null,null,null
6039,Dylan Grant,xsanchez@hotmail.com,1098985454,USNS Edwards
4661,Mary Phillips,vbrown@parker-harrison.net,(732)6615896x309,55677 James Harbors Suite 047
"FPO AP 23850""",null,null,null,null
Lake Bradleychester,"CT 53718""",null,null,null
Cooperton,"AZ 59040""",null,null,null
5684,Gina Koch,brian61@gaines.com,710.459.8498,044 Barnes Loaf Apt. 446
9341,Michelle Carlson,sandersscott@white.com,0836792842x561,06748 Victor Harbors
Tammyville,"VT 90169""",null,null,null


CustomerID,Name,email,phone,Address
4600,Rebecca Yu,dunndaryl@adkins.net,892679475059660,634 Tyler Hollow
Catherinestad,"WV 51848""",null,null,null
6039,Dylan Grant,xsanchez@hotmail.com,1098985454,USNS Edwards
4661,Mary Phillips,vbrown@parker-harrison.net,7326615896309,55677 James Harbors Suite 047
"FPO AP 23850""",null,null,null,null
Lake Bradleychester,"CT 53718""",null,null,null
Cooperton,"AZ 59040""",null,null,null
5684,Gina Koch,brian61@gaines.com,7104598498,044 Barnes Loaf Apt. 446
9341,Michelle Carlson,sandersscott@white.com,0836792842561,06748 Victor Harbors
Tammyville,"VT 90169""",null,null,null


In [0]:
#3.Aggregate sales data like sum of sales by month, customer
 
from pyspark.sql.functions import col, sum, to_date, date_format
 
sales_df1 = sales_df.withColumn("sale_date", to_date(col("SaleDate"), "yyyy-MM-dd"))
display(sales_df1)
 
sales_agg_df = sales_df1.groupBy(date_format(col("sale_date"), "yyyy-MM").alias("month"), "CustomerID").agg(sum("SaleAmount").alias("total_sales"))
display(sales_agg_df)

CustomerID,SaleDate,SaleAmount,sale_date
9341,2023-03-29,413.02,2023-03-29
9341,2023-12-08,245.15,2023-12-08
9829,2023-08-11,141.26,2023-08-11
6647,2023-10-22,203.01,2023-10-22
6647,2023-12-18,213.34,2023-12-18
2453,2023-12-15,223.04,2023-12-15
8689,2023-04-14,33.98,2023-04-14
8689,2023-01-09,312.35,2023-01-09
2453,2023-12-28,207.8,2023-12-28
6647,2023-11-18,143.98,2023-11-18


month,CustomerID,total_sales
2023-11,4600,466.14
2023-11,6647,143.98
2023-08,6039,322.79
2023-04,4600,419.13
2023-02,4661,60.19
2023-02,6039,59.16
2023-04,5684,25.79
2023-08,8689,386.04
2023-06,9829,344.5
2023-12,2453,475.01000000000005


In [0]:
#4.Finding previous month sales by month, customer
from pyspark.sql.window import Window
from pyspark.sql.functions import lag
 
windowSpec = Window.partitionBy("CustomerID").orderBy("month")
 
sales_agg_df1 = sales_agg_df.withColumn("prev_month_sales", lag("total_sales", 1).over(windowSpec))
display(sales_agg_df1)

month,CustomerID,total_sales,prev_month_sales
2023-03,2453,400.89,null
2023-05,2453,357.02,400.89
2023-07,2453,196.79,357.02
2023-08,2453,208.3,196.79
2023-12,2453,475.01000000000005,208.3
2023-03,4600,199.71,null
2023-04,4600,419.13,199.71
2023-06,4600,151.02,419.13
2023-11,4600,466.14,151.02
2023-02,4661,60.19,null


In [0]:
#5.Finding same month last year sales
sales_agg_df2 = sales_agg_df1.withColumn("same_month_last_year_sales", lag("total_sales", 12).over(windowSpec))
display(sales_agg_df2)

month,CustomerID,total_sales,prev_month_sales,same_month_last_year_sales
2023-03,2453,400.89,null,null
2023-05,2453,357.02,400.89,null
2023-07,2453,196.79,357.02,null
2023-08,2453,208.3,196.79,null
2023-12,2453,475.01000000000005,208.3,null
2023-03,4600,199.71,null,null
2023-04,4600,419.13,199.71,null
2023-06,4600,151.02,419.13,null
2023-11,4600,466.14,151.02,null
2023-02,4661,60.19,null,null
